In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from pprint import pformat

from hloc import (
    extract_features,
    match_features,
    pairs_from_covisibility,
    pairs_from_retrieval,
)
from hloc import colmap_from_nvm, triangulation, localize_sfm, visualization

##### 声明必要的输入输出路径

In [3]:
dataset = Path("datasets/")
images = dataset / "ours/"
    
outputs = Path("outputs/ours/")  # where everything will be saved

sfm_pairs = outputs / "pairs-ourdb-covis20.txt"  # top 20 most covisible in SIFT model
loc_pairs = outputs / "pairs-query-netvlad20.txt"  # top 20 retrieved by NetVLAD
reference_sfm = outputs / "sfm_superpoint+superglue"  # the SfM model we will build
results = outputs / "Ours_hloc_superpoint+superglue_netvlad20.txt"  # the result file

# list the standard configurations available
print(f"Configs for feature extractors:\n{pformat(extract_features.confs)}")
print(f"Configs for feature matchers:\n{pformat(match_features.confs)}")

Configs for feature extractors:
{'aliked-n16': {'model': {'model_name': 'aliked-n16', 'name': 'aliked'},
                'output': 'feats-aliked-n16',
                'preprocessing': {'grayscale': False, 'resize_max': 1024}},
 'd2net-ss': {'model': {'multiscale': False, 'name': 'd2net'},
              'output': 'feats-d2net-ss',
              'preprocessing': {'grayscale': False, 'resize_max': 1600}},
 'dir': {'model': {'name': 'dir'},
         'output': 'global-feats-dir',
         'preprocessing': {'resize_max': 1024}},
 'disk': {'model': {'max_keypoints': 5000, 'name': 'disk'},
          'output': 'feats-disk',
          'preprocessing': {'grayscale': False, 'resize_max': 1600}},
 'eigenplaces': {'model': {'name': 'eigenplaces'},
                 'output': 'global-feats-eigenplaces',
                 'preprocessing': {'resize_max': 1024}},
 'netvlad': {'model': {'name': 'netvlad'},
             'output': 'global-feats-netvlad',
             'preprocessing': {'resize_max': 1024}},
 

In [4]:
retrieval_conf = extract_features.confs["netvlad"]
feature_conf = extract_features.confs["superpoint_aachen"]
matcher_conf = match_features.confs["superglue"]

In [5]:
features = extract_features.main(feature_conf, images, outputs)

[2026/01/29 14:40:19 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-r1024',
 'preprocessing': {'grayscale': True, 'resize_max': 1024}}
[2026/01/29 14:40:19 hloc INFO] Found 2091 images in root datasets/ours.
[2026/01/29 14:40:19 hloc INFO] Found 2091 images in root datasets/ours.
[2026/01/29 14:40:19 hloc INFO] Skipping the extraction.
[2026/01/29 14:40:19 hloc INFO] Skipping the extraction.


In [6]:
pairs_from_covisibility.main(outputs / "sfm_sift", sfm_pairs, num_matched=20)

[2026/01/29 14:40:19 hloc INFO] Reading the COLMAP model...
[2026/01/29 14:40:26 hloc INFO] Extracting image pairs from covisibility info...
100%|██████████| 1760/1760 [00:16<00:00, 105.38it/s]
[2026/01/29 14:40:43 hloc INFO] Found 35200 pairs.
100%|██████████| 1760/1760 [00:16<00:00, 105.38it/s]
[2026/01/29 14:40:43 hloc INFO] Found 35200 pairs.


[('cameraImage_100.jpg', 'cameraImage_99.jpg'), ('cameraImage_100.jpg', 'cameraImage_98.jpg'), ('cameraImage_100.jpg', 'cameraImage_97.jpg'), ('cameraImage_100.jpg', 'cameraImage_96.jpg'), ('cameraImage_100.jpg', 'cameraImage_95.jpg'), ('cameraImage_100.jpg', 'cameraImage_93.jpg'), ('cameraImage_100.jpg', 'cameraImage_356.jpg'), ('cameraImage_100.jpg', 'cameraImage_91.jpg'), ('cameraImage_100.jpg', 'cameraImage_355.jpg'), ('cameraImage_100.jpg', 'cameraImage_92.jpg')]


In [7]:
sfm_matches = match_features.main(
    matcher_conf, sfm_pairs, feature_conf["output"], outputs
)


[2026/01/29 14:40:43 hloc INFO] Matching local features with configuration:
{'model': {'name': 'superglue',
           'sinkhorn_iterations': 50,
           'weights': 'outdoor'},
 'output': 'matches-superglue'}
[2026/01/29 14:40:43 hloc INFO] Skipping the matching.
[2026/01/29 14:40:43 hloc INFO] Skipping the matching.


## Triangulate a new SfM model from the given poses
We triangulate the sparse 3D pointcloud given the matches and the reference poses stored in the SIFT COLMAP model.

In [ ]:
# Custom triangulation step to handle "db/" prefix mismatch
# The reference SFM model has names like "cameraImage_100.jpg"
# But our pairs and features use "db/cameraImage_100.jpg"
import pycolmap

reference_model_path = outputs / "sfm_sift"
database_path = reference_sfm / "database.db"
reference_sfm.mkdir(exist_ok=True, parents=True)

# 1. Load reference model
ref_recon = pycolmap.Reconstruction(reference_model_path)

# 2. Create a new database, but manually fix the image names to have "db/" prefix
from hloc.utils.database import COLMAPDatabase
from hloc.triangulation import  import_features, import_matches, estimation_and_geometric_verification, geometric_verification, run_triangulation

if database_path.exists():
    database_path.unlink()

db = COLMAPDatabase.connect(database_path)
db.create_tables()

# Add cameras from reference
for i, camera in ref_recon.cameras.items():
    db.add_camera(
        camera.model.value,
        camera.width,
        camera.height,
        camera.params,
        camera_id=i,
        prior_focal_length=True,
    )

# Add images from reference, BUT PREPEND "db/" to the name
image_ids = {}
for i, image in ref_recon.images.items():
    new_name = "db/" + image.name
    db.add_image(new_name, image.camera_id, image_id=i)
    image_ids[new_name] = i

db.commit()
db.close()

# 3. Continue with standard import features and matches
import_features(image_ids, database_path, features)
import_matches(
    image_ids,
    database_path,
    sfm_pairs,
    sfm_matches,
    min_match_score=None,
    skip_geometric_verification=False,
)

# 4. Update reference reconstruction names in memory to match database!
# This is crucial because triangulation uses names to link the two
for _, image in ref_recon.images.items():
    image.name = "db/" + image.name

# 5. Run geometric verification
geometric_verification(
    image_ids, ref_recon, database_path, features, sfm_pairs, sfm_matches
)

# 6. Run triangulation
# Note: Now `ref_recon` also has "db/" prefixes, so it matches the database.
reconstruction = run_triangulation(
    reference_sfm, database_path, images, ref_recon, verbose=True
)

## Find image pairs via image retrieval
We extract global descriptors with NetVLAD and find for each image the $k$ most similar ones. A larger $k$ improves the robustness of the localization for difficult queries but makes the matching more expensive. Using $k{=}10{-}20$ is generally a good tradeoff but $k{=}50$ gives the best results for the Aachen Day-Night dataset.

In [9]:
global_descriptors = extract_features.main(retrieval_conf, images, outputs)
pairs_from_retrieval.main(
    global_descriptors, loc_pairs, num_matched=20, db_prefix="db", query_prefix="query"
)

[2026/01/29 14:43:22 hloc INFO] Extracting local features with configuration:
{'model': {'name': 'netvlad'},
 'output': 'global-feats-netvlad',
 'preprocessing': {'resize_max': 1024}}
[2026/01/29 14:43:22 hloc INFO] Found 2091 images in root datasets/ours.
[2026/01/29 14:43:22 hloc INFO] Found 2091 images in root datasets/ours.


Using device: cuda


100%|██████████| 2091/2091 [02:19<00:00, 15.02it/s]
[2026/01/29 14:45:46 hloc INFO] Finished exporting features.
[2026/01/29 14:45:46 hloc INFO] Extracting image pairs from a retrieval database.
100%|██████████| 2091/2091 [02:19<00:00, 15.02it/s]
[2026/01/29 14:45:46 hloc INFO] Finished exporting features.
[2026/01/29 14:45:46 hloc INFO] Extracting image pairs from a retrieval database.


Using device: cuda


[2026/01/29 14:45:47 hloc INFO] Found 6620 pairs.


## Match the query images

In [10]:
loc_matches = match_features.main(
    matcher_conf, loc_pairs, feature_conf["output"], outputs
)

[2026/01/29 14:45:47 hloc INFO] Matching local features with configuration:
{'model': {'name': 'superglue',
           'sinkhorn_iterations': 50,
           'weights': 'outdoor'},
 'output': 'matches-superglue'}


Using device: cuda
Loaded SuperGlue model ("outdoor" weights)


/home/mark50/hloc_ws/Hierarchical-Localization/hloc/matchers/../../third_party/SuperGluePretrainedNetwork/models/superglue.py:226: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature

## Localize

In [15]:
import os
import glob

def write_query_intrinsics(output_path, intrinsics_dict, queries_folder, rel_root):
    """
    Recursively find all images in queries_folder and write their intrinsics to output_path.
    intrinsics_dict should contain: model_type, width, height, params (list/tuple)
    """
    # 递归寻找所有图片文件
    query_images = []
    for ext in ["*.jpg", "*.png", "*.jpeg"]:
        query_images.extend(list(queries_folder.rglob(ext)))

    lines = []
    # Convert params to string list once
    params_str = [str(p) for p in intrinsics_dict["params"]]
    
    for p in query_images:
        # Get path relative to the dataset root
        rel_path = p.relative_to(rel_root).as_posix()
        
        # Construct the line
        line = f"{rel_path} {intrinsics_dict['model_type']} {intrinsics_dict['width']} {intrinsics_dict['height']} {' '.join(params_str)}"
        lines.append(line)

    with open(output_path, "w") as f:
        f.write("\n".join(lines))
    print(f"Generated {len(lines)} query intrinsics entries to {output_path}")

def generate_intrinsics_from_db(ref_recon, queries_folder, output_path, dataset_root):
    # Use the first available camera from the reference reconstruction
    first_cam_id = min(ref_recon.cameras.keys())
    first_cam = ref_recon.cameras[first_cam_id]
    
    camera_intrinsics = {
        "model_type": first_cam.model.name,
        "width": first_cam.width,
        "height": first_cam.height,
        "params": first_cam.params,
    }
    write_query_intrinsics(output_path, camera_intrinsics, queries_folder, dataset_root)

def generate_intrinsics_from_self_calib(queries_folder, output_path, dataset_root):
    # Hardcoded self-calibration parameters
    camera_intrinsics = {
        "model_type": "PINHOLE",
        "width": 2448,
        "height": 2048,
        "params": [2407.85558, 2401.83009, 1229.04352, 1037.15274],
    }
    write_query_intrinsics(output_path, camera_intrinsics, queries_folder, dataset_root)

# Setup paths
queries_dir = images / "query/"
output_db_intrinsics = queries_dir / "other_time_period/queries_intrinsics_from_db_camera.txt"
output_self_intrinsics = queries_dir / "other_time_period/queries_intrinsics_from_self_calib.txt"

# 1. Generate from DB camera
# generate_intrinsics_from_db(ref_recon, queries_dir, output_db_intrinsics, images)

# 2. Generate from Self-Calib parameters
generate_intrinsics_from_self_calib(queries_dir, output_self_intrinsics, images)

Generated 331 query intrinsics entries to datasets/ours/query/other_time_period/queries_intrinsics_from_self_calib.txt


In [ ]:
localize_sfm.main(
    reconstruction,
    images / "query/other_time_period/queries_intrinsics_from_db_camera.txt",
    loc_pairs,
    features,
    loc_matches,
    results,
    covisibility_clustering=False,
)  # not required with SuperPoint+SuperGlue

[2026/01/29 16:02:17 hloc.utils.parsers INFO] Imported 331 images from queries_intrinsics_from_self_calib.txt
[2026/01/29 16:02:17 hloc INFO] Reading the 3D model...
[2026/01/29 16:02:17 hloc INFO] Starting localization...
[2026/01/29 16:02:17 hloc INFO] Reading the 3D model...
[2026/01/29 16:02:17 hloc INFO] Starting localization...


Localization config:
{'estimation': {'ransac': {'max_error': 12}}}


100%|██████████| 331/331 [00:41<00:00,  8.03it/s]
[2026/01/29 16:02:58 hloc INFO] Localized 331 / 331 images.
[2026/01/29 16:02:58 hloc INFO] Writing poses to outputs/ours/Ours_hloc_superpoint+superglue_netvlad20.txt...
[2026/01/29 16:02:58 hloc INFO] Writing logs to outputs/ours/Ours_hloc_superpoint+superglue_netvlad20.txt_logs.pkl...
100%|██████████| 331/331 [00:41<00:00,  8.03it/s]
[2026/01/29 16:02:58 hloc INFO] Localized 331 / 331 images.
[2026/01/29 16:02:58 hloc INFO] Writing poses to outputs/ours/Ours_hloc_superpoint+superglue_netvlad20.txt...
[2026/01/29 16:02:58 hloc INFO] Writing logs to outputs/ours/Ours_hloc_superpoint+superglue_netvlad20.txt_logs.pkl...
[2026/01/29 16:02:58 hloc INFO] Done!
[2026/01/29 16:02:58 hloc INFO] Done!
